# 3D Bellman 0827

A staged, visually validated build of the simplified 3D LOS and Bellman problem. Geometry, scenario data, and visualization are maintained in separate Python modules so later terrain models can replace the plane-and-cube map without rewriting downstream logic.

In [23]:
from pathlib import Path
import sys
import numpy as np

# Support both opening Jupyter inside 3D_0827 and executing from the repo root.
module_dir = Path.cwd()
if not (module_dir / 'map_geometry.py').exists():
    module_dir = module_dir / '3D_0827'
if not (module_dir / 'map_geometry.py').exists():
    raise FileNotFoundError('Could not locate the 3D_0827 module directory')
module_path = str(module_dir.resolve())
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# Local modules change during this staged notebook build. Remove any stale
# in-memory versions so rerunning this cell does not require a kernel restart.
local_module_names = (
    'los_explorer_gui', 'visualization', 'reachability_surface',
    'glide_reachability', 'energy_model', 'terrain_catalog',
    'los_geometry', 'ray_tracing', 'scenario', 'map_geometry',
)
for module_name in local_module_names:
    sys.modules.pop(module_name, None)

from los_geometry import (
    build_los_tangent_surface,
    trace_terrain_tangent_contour,
)
from map_geometry import CubeObstacle, MapBounds, PlaneCubeMap
from los_explorer_gui import create_energy_explorer, create_los_explorer
from scenario import MissionPoints, Point3D
from visualization import (
    plot_los_tangent_surface,
    plot_mission_scenario,
    plot_tangent_los_rays,
    plot_terrain_map,
)

## Stage 1 - Empty plane with one cube

Geometry is expressed in compact map units and can be changed here without changing the geometry or plotting modules. Stage 6 applies an explicit physical conversion for energy calculations.

In [24]:
terrain_map = PlaneCubeMap(
    bounds=MapBounds(
        x_min=-10.0,
        x_max=10.0,
        y_min=-8.0,
        y_max=8.0,
    ),
    cube=CubeObstacle(
        center_x=0.0,
        center_y=0.0,
        side_length=4.0,
        base_z=0.0,
    ),
    ground_z=0.0,
)

ground_mesh, cube_mesh = terrain_map.surface_meshes()
print(f'Ground mesh: {len(ground_mesh.vertices)} vertices, {len(ground_mesh.triangles)} triangles')
print(f'Cube mesh:   {len(cube_mesh.vertices)} vertices, {len(cube_mesh.triangles)} triangles')
print(f'Cube z-range: [{terrain_map.cube.base_z:.1f}, {terrain_map.cube.top_z:.1f}] m')

Ground mesh: 4 vertices, 2 triangles
Cube mesh:   8 vertices, 12 triangles
Cube z-range: [0.0, 4.0] m


In [25]:
stage_01_figure = plot_terrain_map(terrain_map)
stage_01_figure.show()

## Stage 2 - Sensor, start point, and goal point

The three mission points are configured independently from the map. The sensor is placed at `(5, 0, 0)` as the fixed origin for the tangent LOS construction.

In [26]:
mission_points = MissionPoints(
    sensor=Point3D(x=5.0, y=0.0, z=0.0),
    start=Point3D(x=-8.0, y=0.0, z=0.0),
    goal=Point3D(x=8.0, y=0.0, z=0.0),
)
mission_points.validate_against(terrain_map)

print(f'Sensor: {mission_points.sensor.as_array()} m')
print(f'Start:  {mission_points.start.as_array()} m')
print(f'Goal:   {mission_points.goal.as_array()} m')

Sensor: [5. 0. 0.] m
Start:  [-8.  0.  0.] m
Goal:   [8. 0. 0.] m


### Stage 2 interactive visual confirmation

Drag to rotate, scroll to zoom, and hover over the sensor, start, goal, plane, and cube to inspect coordinates.

In [27]:
stage_02_figure = plot_mission_scenario(terrain_map, mission_points)
stage_02_figure.show()

### Stage 2 checkpoint

Sensor, start, and goal coordinates used by the Stage 3 tangent-ray calculation.

## Stage 3 - Ten terrain-tangent LOS rays

A dense 2D angular grid is ray-traced against the complete map mesh. Hit/non-hit transitions are refined first; candidates attached to the ground are then removed and the remaining upper/lateral terrain horizon is retained as an open contour. Only then are ten rays selected as visualization samples. No cube edge or tangent height is prescribed.

In [28]:
tangent_contour = trace_terrain_tangent_contour(
    terrain_map.surface_meshes(),
    mission_points.sensor,
    target_mesh_index=1,  # surface_meshes(): ground=0, cube=1
    ground_height=terrain_map.ground_z,
    probe_grid_size=121,
    boundary_refinement_steps=24,
)
visualization_rays = tangent_contour.sample_for_visualization(10)

print(f'Internal probe rays: {tangent_contour.probe_ray_count}')
print(f'Dense refined contour rays: {len(tangent_contour.rays)}')
print(f'Displayed LOS samples: {len(visualization_rays.rays)}')
for index, ray in enumerate(visualization_rays.rays, start=1):
    point = ray.tangent_point
    print(
        f'Ray {index:02d}: tangent=({point.x: .3f}, {point.y: .3f}, {point.z: .3f}) m, '
        f'length={ray.length:.3f} m'
    )

tangent_points = visualization_rays.tangent_points
edge_tolerance = 1.0e-5
left_side_count = np.count_nonzero(np.abs(tangent_points[:, 1] + 2.0) < edge_tolerance)
right_side_count = np.count_nonzero(np.abs(tangent_points[:, 1] - 2.0) < edge_tolerance)
top_count = np.count_nonzero(np.abs(tangent_points[:, 2] - 4.0) < edge_tolerance)
ground_level_count = np.count_nonzero(tangent_points[:, 2] <= terrain_map.ground_z + 1.0e-6)
assert left_side_count > 0 and right_side_count > 0
assert top_count > 0 and ground_level_count == 0
assert not tangent_contour.closed
print(
    'Retained upper-horizon samples: '
    f'left={left_side_count}, right={right_side_count}, '
    f'top={top_count}, ground-level={ground_level_count}; '
    f'discarded ground candidates={tangent_contour.discarded_ground_candidate_count}'
)

Internal probe rays: 14641
Dense refined contour rays: 255
Displayed LOS samples: 10
Ray 01: tangent=( 2.000,  2.000,  0.026) m, length=3.606 m
Ray 02: tangent=( 2.000,  2.000,  1.034) m, length=3.751 m
Ray 03: tangent=( 2.000,  2.000,  2.198) m, length=4.223 m
Ray 04: tangent=( 2.000,  2.000,  3.799) m, length=5.238 m
Ray 05: tangent=( 2.000,  0.644,  4.000) m, length=5.041 m
Ray 06: tangent=( 2.000, -0.644,  4.000) m, length=5.041 m
Ray 07: tangent=( 2.000, -2.000,  3.799) m, length=5.238 m
Ray 08: tangent=( 2.000, -2.000,  2.198) m, length=4.223 m
Ray 09: tangent=( 2.000, -2.000,  1.034) m, length=3.751 m
Ray 10: tangent=( 2.000, -2.000,  0.026) m, length=3.606 m
Retained upper-horizon samples: left=4, right=4, top=2, ground-level=0; discarded ground candidates=85


### Stage 3 interactive visual confirmation

Cyan segments are ten visualization samples from the ray-traced tangent contour. The open magenta horizon includes only elevated top and lateral terrain tangencies. Ground-contact candidates and the artificial bottom closing segment are excluded.

In [29]:
stage_03_figure = plot_tangent_los_rays(
    terrain_map,
    mission_points,
    tangent_contour,
    visualization_rays,
)
stage_03_figure.show()

### Stage 3 checkpoint

Ten finite visualization overlays sampled from the dense contour. Stage 4 uses the complete dense contour, not these ten displayed rays.

## Stage 4 - Infinite extension and LOS tangent surface

The actual surface is `S(s, scale) = sensor + scale * tangent_vector(s)` for open contour parameter `s in [0, 1]` and unbounded `scale >= 0`. All retained upper-horizon rays define the display mesh; the ten cyan rays remain visualization overlays only. Scale 4 is merely the Plotly clipping distance. No panel reconnects the two contour endpoints across the terrain base.

In [30]:
los_tangent_surface = build_los_tangent_surface(
    tangent_contour,
    display_extension_factor=4.0,
)
surface_mesh = los_tangent_surface.display_mesh()

assert len(visualization_rays.rays) == 10
alternate_visualization_rays = tangent_contour.sample_for_visualization(25)
assert len(alternate_visualization_rays.rays) == 25
expected_panel_count = len(tangent_contour.rays) - (0 if tangent_contour.closed else 1)
assert los_tangent_surface.panel_count == expected_panel_count
assert los_tangent_surface.panel_count > len(visualization_rays.rays)
assert los_tangent_surface.panel_count > len(alternate_visualization_rays.rays)
assert np.allclose(
    los_tangent_surface.section_points(0.0),
    mission_points.sensor.as_array(),
)
assert np.allclose(
    los_tangent_surface.section_points(1.0),
    tangent_contour.tangent_points,
)

far_points = los_tangent_surface.section_points(
    los_tangent_surface.display_extension_factor
)
test_s = 0.123456
assert np.allclose(
    los_tangent_surface.point_at(test_s, 1.0),
    mission_points.sensor.as_array() + tangent_contour.tangent_vector_at(test_s),
)
assert np.linalg.norm(los_tangent_surface.point_at(test_s, 1.0e6)) > 1.0e6

print('Continuous surface domain: s in [0, 1], scale >= 0 (unbounded, open horizon)')
print('Terrain tangent contour: scale = 1')
print(f'Display clip: scale = {los_tangent_surface.display_extension_factor:g}')
print(f'Displayed sample rays: {len(visualization_rays.rays)}')
print(f'Dense surface panels: {los_tangent_surface.panel_count}')
print('Sample-count independence: 10 -> 25 displayed rays, surface remains unchanged')
print(f'Display mesh: {len(surface_mesh.vertices)} vertices, {len(surface_mesh.triangles)} triangles')
print(f'Far-section z-range: [{far_points[:, 2].min():.3f}, {far_points[:, 2].max():.3f}] m')

Continuous surface domain: s in [0, 1], scale >= 0 (unbounded, open horizon)
Terrain tangent contour: scale = 1
Display clip: scale = 4
Displayed sample rays: 10
Dense surface panels: 254
Sample-count independence: 10 -> 25 displayed rays, surface remains unchanged
Display mesh: 256 vertices, 254 triangles
Far-section z-range: [0.105, 16.000] m


### Stage 4 interactive visual confirmation

The translucent cyan geometry is a dense display mesh of the continuous unbounded surface. Only ten cyan/blue ray lines are drawn as visual samples. The dark outer line is the finite Plotly clip and is not a physical surface boundary.

In [31]:
stage_04_figure = plot_los_tangent_surface(
    terrain_map,
    mission_points,
    los_tangent_surface,
    visualization_rays,
)
stage_04_figure.show()

### Stage 4 checkpoint

Confirm that the dense surface is visually continuous while only ten sample rays are emphasized, and that top/lateral propagation is present before beginning total-energy computation.

## Stage 5 - Interactive terrain and defender-position GUI

Use the terrain toggle and the two sensor-position sliders to regenerate the complete ray-traced LOS region. Sensor `x` is restricted to `[5, 10]`; sensor `y` spans the full map width `[-8, 8]`. Slider updates run when the handle is released. The ray tracer retains only the open upper/lateral terrain horizon; ground-contact tangent candidates are discarded. Total-energy and goal-reachable overlays are intentionally deferred until these geometry cases are approved.

In [32]:
los_explorer = create_los_explorer(
    probe_grid_size=101,
    displayed_ray_count=10,
)
los_explorer.display()

### Stage 5 checkpoint

Check every terrain toggle and move both sensor sliders through representative positions. The terrain, defender marker, dense tangent contour, continuous LOS surface, and ten displayed sample rays should update together.

## Stage 6 - Total energy and goal-reachable LOS regions

Each point on the displayed ground-trimmed LOS tangent surface is treated as a powered-to-glide switching candidate. Powered flight follows the straight start-to-switch segment and delivers a full position/velocity state. The glide model retains total mechanical energy, trims to the Ka 6 CR reference speed, preserves horizontal heading, and evaluates a bounded-turn path into the 25 m goal region. Straight-flight `L/D` is deliberately de-rated to 10. During a 30-degree turn, the induced-drag approximation `L/D_turn = L/D / n^2` gives 7.5. A 10 m equivalent-height switching loss is also charged. Green is goal-reachable and red is goal-unreachable; both surfaces use `alpha=0.25` and are clipped exactly at zero interpolated energy margin instead of triangle-majority coloring.

Prototype parameters: Schleicher Ka 6 CR gross mass 304 kg, reference speed 80 km/h, user-selected `L/D=10`, maximum bank 30 degrees, switching loss 10 m, and 100 physical metres per displayed map unit. Powered flight at constant 80 km/h, the de-rated glide ratio, the bank bound, and switching loss are prototype assumptions. Aircraft mass and speed references are taken from the [Belgian Air Accident Investigation Unit report](https://mobilit.belgium.be/sites/default/files/domain/Aviation/Veiligheid/Verslagen%20voorvallen/2010/2010_5.pdf). The finite colored mesh is only a visualization sampling of the continuous, unbounded LOS tangent surface.

In [ ]:
energy_explorer = create_energy_explorer(
    probe_grid_size=101,
    displayed_ray_count=10,
    radial_section_count=25,
    contour_section_count=96,
)
energy_explorer.display()

### Stage 6 checkpoint

Confirm that both red and green regions are present, that green generally occupies the higher-energy portion of the surface, and that terrain/sensor changes recompute both the LOS geometry and energy classification. The status line reports reachable faces, unreachable faces, and powered-infeasible vertices separately.